# Validate a migrated wave

Runs the reconciliation checks in `migration/<wave>/validation/validation.yml`, comparing Talend output (expected) with Databricks output (actual). Fails if any check does not pass.

In [0]:
import os

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_root = "/Workspace" + os.path.dirname(os.path.dirname(_nb_path))

dbutils.widgets.text("wave", "wave_01")
dbutils.widgets.text("root_path", _root)
dbutils.widgets.text("checks", "")  # optional comma-separated check names; empty = all
dbutils.widgets.text("snowflake_schema", "")  # dev shadow schema the jobs wrote to, fills ${snowflake_schema}

wave = dbutils.widgets.get("wave")
root = dbutils.widgets.get("root_path")
check_names = [c.strip() for c in dbutils.widgets.get("checks").split(",") if c.strip()] or None
variables = {"snowflake_schema": dbutils.widgets.get("snowflake_schema")}
config_path = f"{root}/migration/{wave}/validation/validation.yml"
print(config_path)

In [0]:
import json
import sys

if f"{root}/src" not in sys.path:
    sys.path.insert(0, f"{root}/src")

from operators.validation import run_from_config

result = run_from_config(spark, dbutils, config_path, check_names, variables=variables)

In [0]:
failed = [name for name, r in result["checks"].items() if not r["passed"]]
if failed or not result["checks"]:
    raise AssertionError(f"Validation failed for {wave}: {failed or 'no checks ran'}")
dbutils.notebook.exit(json.dumps(result, default=str))